# Voice Matching Benchmark

This notebook benchmarks **speaker verification / voice matching** models for speaker-to-name mapping task.

## Models covered
- **SpeechBrain ECAPA-TDNN** (`speechbrain/spkrec-ecapa-voxceleb`)
- **Microsoft WavLM Base Plus SV** (`microsoft/wavlm-base-plus-sv`)


This notebook works with a ZIP where **each folder name is the speaker name** and files inside are that person's voice recordings.

It can run **without a separate test set** by automatically creating enroll/test splits:

- If a speaker has **2 or more files**:
  - first file -> enrollment
  - remaining files -> test
- If a speaker has **1 long file**:
  - it splits the file into:
    - enrollment chunk (first 20 sec)
    - test chunk (remainning length)

Supported audio types:
- `.wav`
- `.mp3`
- `.ogg`
- `.m4a`
- `.flac`


## Notes
- This notebook evaluates **matching accuracy**, not diarization.

# Code

In [1]:
!pip -q uninstall -y peft
!pip -q install pandas==2.2.2 librosa soundfile scikit-learn matplotlib tqdm speechbrain transformers==4.41.2 sentencepiece accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 56.2 MB/s eta 0:00:00


In [6]:
import os
import io
import math
import json
import zipfile
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

ROOT = Path("/content/voice_benchmark")
RAW_DIR = ROOT / "raw"
SPLIT_DIR = ROOT / "prepared"
EXPORT_DIR = ROOT / "exports"

for p in [ROOT, RAW_DIR, SPLIT_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TARGET_SR = 16000
ENROLL_SECONDS = 20
GAP_SECONDS = 1
TEST_MIN_SECONDS = 4
TEST_CHUNK_SECONDS = 8

AUDIO_EXTS = {".wav", ".mp3", ".ogg", ".m4a", ".flac"}

print("Setup complete.")

Setup complete.


In [7]:
from google.colab import files

uploaded = files.upload()

zip_path = None
for name, data in uploaded.items():
    if name.lower().endswith(".zip"):
        zip_path = ROOT / name
        with open(zip_path, "wb") as f:
            f.write(data)

if zip_path is None:
    raise ValueError("Please upload your ZIP file.")

if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(RAW_DIR)

print("Uploaded ZIP:", zip_path)
print("Extracted to:", RAW_DIR)

Saving Voice_Data_Sample.zip to Voice_Data_Sample (1).zip
Uploaded ZIP: /content/voice_benchmark/Voice_Data_Sample (1).zip
Extracted to: /content/voice_benchmark/raw


In [8]:
def collect_speaker_dirs(base_dir):
    speaker_map = {}
    for p in base_dir.rglob("*"):
        if p.is_dir():
            files_ = sorted([x for x in p.iterdir() if x.is_file() and x.suffix.lower() in AUDIO_EXTS])
            if files_:
                speaker_map[p.name] = files_
    return speaker_map

speaker_files = collect_speaker_dirs(RAW_DIR)

print("Detected speakers:")
for spk, files_ in speaker_files.items():
    print(f"\n{spk}:")
    for fp in files_:
        print(" -", fp.name)

Detected speakers:

Girl1:
 - Girl1_1.ogg
 - Girl1_2.ogg

g2:
 - g2.ogg

SID:
 - SID_1.ogg
 - SID_2.ogg

ethan:
 - ethan.ogg

AI:
 - AI_1.mp3
 - AI_2.mp3

p1:
 - p1_1.ogg


In [9]:
def load_audio(path, sr=TARGET_SR):
    audio, _ = librosa.load(path, sr=sr, mono=True)
    if audio.size == 0:
        raise ValueError(f"Empty audio: {path}")
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    return audio.astype(np.float32), sr

def save_wav(path, audio, sr=TARGET_SR):
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), audio, sr)

def l2_normalize(x, eps=1e-10):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    n = np.linalg.norm(x)
    return x / max(n, eps)

def average_embeddings(embs):
    embs = np.stack([l2_normalize(e) for e in embs], axis=0)
    return l2_normalize(np.mean(embs, axis=0))

if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)
(SPLIT_DIR / "enroll").mkdir(parents=True, exist_ok=True)
(SPLIT_DIR / "test").mkdir(parents=True, exist_ok=True)

labels_rows = []
summary_rows = []

for speaker, files_ in speaker_files.items():
    enroll_spk_dir = SPLIT_DIR / "enroll" / speaker
    test_spk_dir = SPLIT_DIR / "test" / speaker
    enroll_spk_dir.mkdir(parents=True, exist_ok=True)
    test_spk_dir.mkdir(parents=True, exist_ok=True)

    if len(files_) >= 2:
        # Strict rule: first file only for enroll, never seen in test
        first = files_[0]
        y, sr = load_audio(first)
        save_wav(enroll_spk_dir / f"{speaker}_enroll_1.wav", y, sr)

        test_count = 0
        for i, fp in enumerate(files_[1:], start=1):
            y, sr = load_audio(fp)
            out = test_spk_dir / f"{speaker}_test_{i}.wav"
            save_wav(out, y, sr)
            labels_rows.append({"file": out.name, "speaker": speaker})
            test_count += 1

        summary_rows.append({
            "speaker": speaker,
            "strategy": "first_file_enroll_rest_test",
            "num_raw_files": len(files_),
            "num_enroll_files": 1,
            "num_test_files": test_count,
        })

    elif len(files_) == 1:
        fp = files_[0]
        y, sr = load_audio(fp)
        total_sec = len(y) / sr

        if total_sec <= ENROLL_SECONDS + GAP_SECONDS + TEST_MIN_SECONDS:
            print(f"Skipping {speaker}: single file too short ({total_sec:.1f}s) for strict 20s enroll + test split.")
            continue

        enroll_len = int(ENROLL_SECONDS * sr)
        gap_len = int(GAP_SECONDS * sr)

        enroll_audio = y[:enroll_len]
        save_wav(enroll_spk_dir / f"{speaker}_enroll_1.wav", enroll_audio, sr)

        remaining = y[enroll_len + gap_len:]
        chunk_len = int(TEST_CHUNK_SECONDS * sr)

        test_count = 0
        pos = 0
        while pos < len(remaining):
            chunk = remaining[pos:pos + chunk_len]
            if len(chunk) < int(TEST_MIN_SECONDS * sr):
                break
            test_count += 1
            out = test_spk_dir / f"{speaker}_test_{test_count}.wav"
            save_wav(out, chunk, sr)
            labels_rows.append({"file": out.name, "speaker": speaker})
            pos += chunk_len

        summary_rows.append({
            "speaker": speaker,
            "strategy": "single_file_first_20s_enroll_rest_test",
            "num_raw_files": 1,
            "num_enroll_files": 1,
            "num_test_files": test_count,
        })

labels_df = pd.DataFrame(labels_rows)
split_summary_df = pd.DataFrame(summary_rows)

print("Split summary:")
display(split_summary_df)

print("\nGenerated test files:")
display(labels_df.head(100))

Split summary:


,speaker,strategy,num_raw_files,num_enroll_files,num_test_files
0,Girl1,first_file_enroll_rest_test,2,1,1
1,g2,single_file_first_20s_enroll_rest_test,1,1,2
2,SID,first_file_enroll_rest_test,2,1,1
3,ethan,single_file_first_20s_enroll_rest_test,1,1,1
4,AI,first_file_enroll_rest_test,2,1,1
5,p1,single_file_first_20s_enroll_rest_test,1,1,2



Generated test files:


,file,speaker
0,Girl1_test_1.wav,Girl1
1,g2_test_1.wav,g2
2,g2_test_2.wav,g2
3,SID_test_1.wav,SID
4,ethan_test_1.wav,ethan
5,AI_test_1.wav,AI
6,p1_test_1.wav,p1
7,p1_test_2.wav,p1


In [10]:
#Eload ECAPA and WavLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ECAPA
from speechbrain.inference.speaker import EncoderClassifier

ecapa_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": device},
)

def emb_ecapa(audio, sr=TARGET_SR):
    wav = torch.tensor(audio).unsqueeze(0)
    with torch.no_grad():
        emb = ecapa_model.encode_batch(wav)
    return emb.squeeze().detach().cpu().numpy()

# WavLM
from transformers import Wav2Vec2FeatureExtractor, WavLMForXVector

wavlm_fe = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
wavlm_model = WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv").to(device)
wavlm_model.eval()

def emb_wavlm(audio, sr=TARGET_SR):
    inputs = wavlm_fe(audio, sampling_rate=sr, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = wavlm_model(**inputs).embeddings
    emb = torch.nn.functional.normalize(out, dim=-1).squeeze().detach().cpu().numpy()
    return emb

MODEL_FNS = {
    "ecapa": emb_ecapa,
    "wavlm_plus_sv": emb_wavlm,
}

print("Models loaded:", list(MODEL_FNS.keys()))

Using device: cuda


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


hyperparams.yaml: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/405M [00:00<?, ?B/s]

Some weights of the model checkpoint at microsoft/wavlm-base-plus-sv were not used when initializing WavLMForXVector: ['wavlm.encoder.pos_conv_embed.conv.weight_g', 'wavlm.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing WavLMForXVector from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing WavLMForXVector from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of WavLMForXVector were not initialized from the model checkpoint at microsoft/wavlm-base-plus-sv and are newly initialized: ['wavlm.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wavlm.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a d

Models loaded: ['ecapa', 'wavlm_plus_sv']


In [11]:
#helpers
def build_prototypes(embed_fn):
    enroll_root = SPLIT_DIR / "enroll"
    prototypes = {}

    for speaker_dir in sorted([p for p in enroll_root.iterdir() if p.is_dir()]):
        embs = []
        for wav_path in sorted(speaker_dir.glob("*.wav")):
            audio, sr = load_audio(wav_path)
            emb = embed_fn(audio, sr)
            embs.append(emb)
        if embs:
            prototypes[speaker_dir.name] = average_embeddings(embs)

    return prototypes

def evaluate_model(model_name, embed_fn):
    prototypes = build_prototypes(embed_fn)
    rows = []

    for speaker_dir in sorted([p for p in (SPLIT_DIR / "test").iterdir() if p.is_dir()]):
        for wav_path in sorted(speaker_dir.glob("*.wav")):
            true_speaker = speaker_dir.name
            audio, sr = load_audio(wav_path)
            emb = l2_normalize(embed_fn(audio, sr))

            names = list(prototypes.keys())
            sims = [float(cosine_similarity([emb], [prototypes[n]])[0][0]) for n in names]
            ranked = sorted(zip(names, sims), key=lambda x: x[1], reverse=True)
            pred = ranked[0][0]

            rows.append({
                "model": model_name,
                "file": wav_path.name,
                "true_speaker": true_speaker,
                "pred_speaker": pred,
                "top1_score": ranked[0][1],
                "top2_score": ranked[1][1] if len(ranked) > 1 else np.nan,
                "gap_top1_top2": (ranked[0][1] - ranked[1][1]) if len(ranked) > 1 else np.nan,
                "ranking": ranked,
            })

    pred_df = pd.DataFrame(rows)
    acc = accuracy_score(pred_df["true_speaker"], pred_df["pred_speaker"])
    return acc, pred_df

print("Benchmark helpers ready.")

Benchmark helpers ready.


In [12]:
results = {}

for model_name, embed_fn in MODEL_FNS.items():
    print(f"\n=== Evaluating {model_name} ===")
    acc, pred_df = evaluate_model(model_name, embed_fn)
    results[model_name] = {
        "accuracy": acc,
        "predictions": pred_df,
    }
    print(f"Accuracy: {acc:.4f}")
    display(pred_df[["file", "true_speaker", "pred_speaker", "top1_score", "top2_score", "gap_top1_top2"]])


=== Evaluating ecapa ===
Accuracy: 1.0000


,file,true_speaker,pred_speaker,top1_score,top2_score,gap_top1_top2
0,AI_test_1.wav,AI,AI,0.994072,0.194571,0.799500
1,Girl1_test_1.wav,Girl1,Girl1,0.874730,0.116771,0.757959
2,SID_test_1.wav,SID,SID,0.889987,0.245415,0.644572
3,ethan_test_1.wav,ethan,ethan,0.938323,0.144374,0.793949
4,g2_test_1.wav,g2,g2,0.900725,0.162326,0.738399
5,g2_test_2.wav,g2,g2,0.881240,0.178229,0.703012
6,p1_test_1.wav,p1,p1,0.784711,0.215228,0.569483
7,p1_test_2.wav,p1,p1,0.778182,0.217676,0.560506



=== Evaluating wavlm_plus_sv ===
Accuracy: 1.0000


,file,true_speaker,pred_speaker,top1_score,top2_score,gap_top1_top2
0,AI_test_1.wav,AI,AI,0.998434,0.721398,0.277036
1,Girl1_test_1.wav,Girl1,Girl1,0.983616,0.717561,0.266055
2,SID_test_1.wav,SID,SID,0.975730,0.848810,0.126920
3,ethan_test_1.wav,ethan,ethan,0.983971,0.916645,0.067326
4,g2_test_1.wav,g2,g2,0.982060,0.723357,0.258702
5,g2_test_2.wav,g2,g2,0.987757,0.696082,0.291675
6,p1_test_1.wav,p1,p1,0.974166,0.904786,0.069379
7,p1_test_2.wav,p1,p1,0.954036,0.907702,0.046334


In [13]:
summary_rows = []

for model_name, bundle in results.items():
    df = bundle["predictions"]
    summary_rows.append({
        "model": model_name,
        "accuracy": bundle["accuracy"],
        "avg_top1_score": float(df["top1_score"].mean()),
        "avg_gap_top1_top2": float(df["gap_top1_top2"].dropna().mean()),
        "num_test_clips": len(df),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["accuracy", "avg_gap_top1_top2"],
    ascending=False
)

print("Final comparison:")
display(summary_df)

CONF_THRESHOLD = 0.82
GAP_THRESHOLD = 0.05

best_model = summary_df.iloc[0]["model"]
best_df = results[best_model]["predictions"].copy()

best_df["decision"] = np.where(
    (best_df["top1_score"] >= CONF_THRESHOLD) & (best_df["gap_top1_top2"] >= GAP_THRESHOLD),
    best_df["pred_speaker"],
    "[REVIEW_NEEDED]",
)

print("\nBest model:", best_model)
display(best_df[["file", "true_speaker", "pred_speaker", "top1_score", "gap_top1_top2", "decision"]])

split_summary_df.to_csv(EXPORT_DIR / "split_summary.csv", index=False)
summary_df.to_csv(EXPORT_DIR / "model_summary.csv", index=False)

for model_name, bundle in results.items():
    df = bundle["predictions"].copy()
    df["ranking"] = df["ranking"].astype(str)
    df.to_csv(EXPORT_DIR / f"{model_name}_predictions.csv", index=False)

print("\nSaved files:")
for p in sorted(EXPORT_DIR.glob("*.csv")):
    print("-", p)

Final comparison:


,model,accuracy,avg_top1_score,avg_gap_top1_top2,num_test_clips
0,ecapa,1.0,0.880246,0.695922,8
1,wavlm_plus_sv,1.0,0.979971,0.175429,8



Best model: ecapa


,file,true_speaker,pred_speaker,top1_score,gap_top1_top2,decision
0,AI_test_1.wav,AI,AI,0.994072,0.799500,AI
1,Girl1_test_1.wav,Girl1,Girl1,0.874730,0.757959,Girl1
2,SID_test_1.wav,SID,SID,0.889987,0.644572,SID
3,ethan_test_1.wav,ethan,ethan,0.938323,0.793949,ethan
4,g2_test_1.wav,g2,g2,0.900725,0.738399,g2
5,g2_test_2.wav,g2,g2,0.881240,0.703012,g2
6,p1_test_1.wav,p1,p1,0.784711,0.569483,[REVIEW_NEEDED]
7,p1_test_2.wav,p1,p1,0.778182,0.560506,[REVIEW_NEEDED]



Saved files:
- /content/voice_benchmark/exports/ecapa_predictions.csv
- /content/voice_benchmark/exports/model_summary.csv
- /content/voice_benchmark/exports/split_summary.csv
- /content/voice_benchmark/exports/wavlm_plus_sv_predictions.csv



#Conclusion and Result

## Dataset Used
This benchmark was run on a small internal speaker-verification sample set designed to evaluate
voice-matching quality for speaker-to-name mapping.

- Reported distinct speaker identities used for this write-up: **6**
- Real human voices: **4**
- Synthetic AI voice(s): **1**
- Recordings captured in more realistic / noisy conditions: **2**
- Approximate clip duration per source file: **20–40 seconds**

In this benchmark, the available data included both:
- speakers with **multiple source recordings**, where the first file was used strictly for enrollment and the remaining file(s) were used only for testing
- speakers with **a single longer source recording**, where the first **20 seconds** were used strictly for enrollment and the remaining portion was used only for testing

### Split Notes
- Multi-file speakers: **Girl1, SID, AI**
- Single-file auto-split speakers: **g2, ethan, p1**

## Model Comparison
- **ecapa**: accuracy = **1.0000**, avg top-1 score = **0.8802**, avg top-1/top-2 gap = **0.6959**, test clips = **8**
- **wavlm_plus_sv**: accuracy = **1.0000**, avg top-1 score = **0.9800**, avg top-1/top-2 gap = **0.1754**, test clips = **8**

## Key Result
The best-performing model in this benchmark was **ecapa**.

- Best-model accuracy: **1.0000**
- Average top-1 similarity score: **0.8802**
- Average top-1 vs top-2 separation gap: **0.6959**
- Total evaluated test clips: **8**
- Auto-assigned clips at the current confidence gate: **6**
- Review-needed clips at the current confidence gate: **2**

## Insights
1. **The benchmark indicates that voice matching is feasible on the current sample set.**  
   The best model achieved strong performance and showed usable separation between the top-1 and
   top-2 candidates, which is important for confidence-gated speaker-name assignment.

2. **The confidence-gap signal is as important as raw accuracy.**  
   Even when a model predicts the correct speaker, production use should prefer cases where the
   top match is clearly separated from the second-best match. This reduces the chance of assigning
   the wrong real name in downstream transcripts.

3. **This is a strong first benchmark, but not yet final production validation.**  
   For speakers that had only one source file, enrollment and testing were created from different
   parts of the same original recording. That is still a valid first-pass benchmark, but it is
   easier than fully independent train/test data captured on different days, devices, or noise conditions.

4. **Real-world variability is still the main challenge.**  
   The inclusion of noisy / real-environment samples is useful because it makes the test more relevant
   to actual meeting conditions. However, the benchmark should be expanded further with more speakers,
   more sessions per speaker, and more varied acoustic conditions before locking a final production model.

## Conclusion
Based on this benchmark, **ecapa** is the strongest candidate among the models evaluated in this run
and is the most suitable starting point for the next phase of implementation.

The result suggests that:
- the overall voice-matching direction is valid
- the current pipeline can distinguish speaker identity with promising precision on a small sample set
- a confidence-gated matching strategy remains necessary before assigning real names automatically
